<a href="https://colab.research.google.com/github/wingated/cs473/blob/main/labs/cs473_lab_week_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><p><b>After clicking the "Open in Colab" link, copy the notebook to your own Google Drive before getting started, or it will not save your work</b></p>

# BYU CS 473 Lab Week 6

## Introduction:

The concept of mutual information is powerful in theory, but in practice it can be difficult to estimate. In this lab, you will implement a conceptually simple way to estimate the mutual information between two random variables. It is known as the KSG estimator (from "Kraskov–Stögbauer–Grassberger").

You will then use your estimator as part of a feature selection algorithm.

---
## Grading standards   

Your notebook will be graded on the following:

* 60% Correct implementation of KSG estimator
* 20% Correct implementation of "all_mutual_inf"
* 20% Correct selection of most informative columns

---
## Description

For this lab, we will use this dataset [lab_week_6_data.csv](https://github.com/wingated/cs473/blob/main/labs/data/lab_week_6_data.csv).

Download the csv from the link above, then upload the csv to your current google colab runtime.

Then begin by loading and preparing the dataset:

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv( "lab_week_6_data.csv" )

X = df[['x1','x2','x3','x4','x5']]
y = df['y']

X = np.array(X)
y = np.array(y)


The KSG estimator is an elegant idea that's fairly easy to implement. The idea is to use an adaptive grid, where the grid depends on the local density of the data.  This is accomplished by calculating the k'th nearest neighbor for each point, and then defining a box around each point that just barely extends to include the k'th nearest neighbor. Importantly, the size of the box is also used in the marginal entropy calculations

To get some intuition for the method, check out [this demonstration on Wolfram Alpha's site](https://demonstrations.wolfram.com/KraskovKSGEstimatorOfMutualInformation/).  Here is a screenshot of their UI:

![Wolfram UI](https://raw.githubusercontent.com/wingated/cs473/main/labs/images/lab_week_6_image1.png)

The algorithm is fairly straightforward.  We are given N samples from the joint distribution of p(X,Y). First, we calculate the *local mutual information* for each data point:

1. For each datapoint $d_j = (x_j,y_j)$
2. Find the k'th nearest neighbor (call this point $n_j$) -- you should use the infinity norm (max norm), and distance should be calculated using both coordinates (ie, don't just calculate distances using only the x or y coordinate)
3. Count the number of points in x whose 'x-distance' fall within the distance to the k'th nearest neighbor (calculated above) of $d_j$ (ie, ignore y); call this number $n_{x{_j}}$.  Do the same for the points in y (these are the shaded gray boxes in the image); call it $n_{y{_j}}$.

Then, we calculate the final mutual information by averaging over all of the datapoints and adding a few more digamma calls:

![final mutual information](https://raw.githubusercontent.com/wingated/cs473/main/labs/images/lab_week_6_image3.png)

where $\Psi$ is the digamma function

If you want more details on the implementation, see page 2 of [Estimating Mutual Information](https://arxiv.org/pdf/cond-mat/0305641)

In [ ]:
print(X.shape)
print(y.shape)

(100, 5)
(100,)


In [40]:
from scipy.special import digamma
import numpy as np

def mutual_inf( X, Y, k ):

    N = X.shape[0]
    psi_N = digamma(N)
    psi_k = digamma(k)

    # we need to iterate through every point in the dataset. But we also need
    # to have the points all together. merge them together now.

    # we need to make Y a 100x1 vector
    Y_reshaped = np.reshape(Y, (-1,1))

    merged = np.hstack((X,Y_reshaped))

    sum_digamma_terms = 0.0 # Initialize sum for digamma terms

    #now we are going to have to iterate through every point(column)
    for i in range(N):
      z = merged[i]

      #here we get a vector of distances from z
      distances = np.linalg.norm(merged - z, axis=1, ord=np.inf)

      # here we calculate indices of the k-th nearest neighbor distance (epsilon_i).
      # The k-th nearest neighbor (excluding itself) is at index k in the sorted array.
      sorted_indices = np.argsort(distances)
      epsilon_i = distances[sorted_indices[k]]

      # find the number of points within epsilon_i in x dimension
      current_X_vec = z[0:X.shape[1]]
      n_x = np.sum(np.linalg.norm(X - current_X_vec, axis=1, ord=np.inf) < epsilon_i)

      # find the number of points within epsilon_i in y dimension
      current_y_scalar = z[X.shape[1]]
      n_y = np.sum(np.abs(Y_reshaped.flatten() - current_y_scalar) < epsilon_i)

      # Sum the digamma terms as per the formula
      sum_digamma_terms += (digamma(max(1, n_x)) + digamma(max(1, n_y)))

    # Calculate the final mutual information
    MI = psi_N + psi_k - (1.0/N) * sum_digamma_terms

    return MI

mutual_inf(X,y,3)

np.float64(0.5055504726754698)

---
## Part 2:

Now that you can estimate mutual information between two variables, we're going to use your method in the context of a regression problem.

Given our X,y data from the beginning of the lab, run your function and calculate the mutual information between every column of X and y.  This should result in a list of scores, that might look something like this:

```python
array([1.2641203602210282, 0.07974959311825458, 0.03321433628330617, 0.02213361087954091, 0.0, 0.0, 0.0])
```

If you wrap this in a single function (call it "all_mutual_inf"), then you can use scikit learn's feature selection algorithms:

In [45]:
print(range(X.shape[1]))

range(0, 5)


In [48]:
for j in range(0,20):
  mis = np.array([mutual_inf(X[:, i].reshape(-1, 1), y, j) for i in range(X.shape[1])])
  print(mis)

[-inf -inf -inf -inf -inf]
[ 1.41453528  0.17546116  0.15131076  0.05162396 -0.23595019]
[ 1.49841468  0.20448825  0.19767516  0.16313141 -0.18944217]
[ 1.56167519  0.23807644  0.18881414  0.17276823 -0.08238495]
[ 1.5064027   0.212437    0.16207124  0.16227841 -0.0572508 ]
[ 1.43650412  0.20229339  0.12876745  0.16582399 -0.02112998]
[ 1.36090223  0.19892357  0.13450592  0.16837197 -0.01325521]
[ 1.31319314  0.20767318  0.12154948  0.17151696 -0.00942805]
[ 1.26711776  0.20571906  0.11367862  0.18235149 -0.00400163]
[ 1.205556    0.20390887  0.12108307  0.18533079 -0.00852323]
[1.15434131e+00 2.02967028e-01 1.32616439e-01 1.77218902e-01
 3.23763181e-04]
[1.08829253 0.21671574 0.13721736 0.17727024 0.00473039]
[1.0051051  0.21837629 0.11770472 0.18142391 0.00489518]
[ 0.94645955  0.21060727  0.11337736  0.18696148 -0.00445323]
[9.03148189e-01 2.11407354e-01 1.04241038e-01 1.83937613e-01
 5.65664556e-05]
[8.43812158e-01 2.05325627e-01 1.11621437e-01 1.82685281e-01
 6.35761296e-04]
[ 0.7

In [49]:
def all_mutual_inf( X, y ):
    # return a list of mutual informations
    # the list should be as long as the number of columns in X

    # Using k=4 as an example, consistent with the problem's example call
    k = 4

    mutual_info_scores = []
    num_features = X.shape[1]

    for i in range(num_features):
        # Extract the i-th column of X and reshape to 2D for mutual_inf function
        X_col_i = X[:, i].reshape(-1, 1)
        # Calculate mutual information for this column and y
        mi_score = mutual_inf(X_col_i, y, k)
        mutual_info_scores.append(mi_score)

    return np.array(mutual_info_scores)

all_mutual_inf(X,y)

array([ 1.5064027 ,  0.212437  ,  0.16207124,  0.16227841, -0.0572508 ])

In [51]:
from sklearn.feature_selection import SelectKBest

X_new = SelectKBest( all_mutual_inf, k=2 ).fit_transform(X, y)

So: which X features are most informative about y?

---



The first column of x features have the highest mutual information. This means that they have the most information shared with y, and therefore are most informative about y.

---
## Hints

The following functions may be useful to you:

In [ ]:
scipy.special.digamma

For fun, you could also [read more about the KSG estimator](https://arxiv.org/pdf/1604.03006).